# 01 Org Chart Overhaul

## Your Objective
Unravel this parent-child hierarchy puzzle by mapping each employee's chain of command and calculating their direct and total reports.

Starting with a list of employees and their managers, your task is to create 3 new columns:

Reporting Hierarchy: The chain of command from the highest-ranking manager down to you

Direct Reports: The number of employees that report to you as their manager

Total Reports: The total number of employees beneath you in the reporting hierarchy (your direct reports, plus their direct reports, etc.)

![images](../images/y9sIcNGbdLtOuatPP7OZURa0.avif)

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/OfficeSpace.csv")
df.head()

,Employee Name,Manager Name
0,Bill Lumbergh,NaN
1,Bob Slydell,Bill Lumbergh
2,Bob Porter,Bill Lumbergh
3,Linda M. Grayson,Bill Lumbergh
4,Dom Portwood,Linda M. Grayson


In [3]:
emp_to_mgr = {}
mgr_to_rep = {}

for _, row in df.iterrows():
    emp = row['Employee Name']
    mgr = row['Manager Name']

    emp_to_mgr[emp] = mgr if pd.notna(mgr) else None

    if pd.notna(mgr):
        mgr_to_rep.setdefault(mgr, []).append(emp)

df["Direct Reports Count"] = df["Employee Name"].apply(lambda x: len(mgr_to_rep[x]) if x in mgr_to_rep else 0)

df.head()

,Employee Name,Manager Name,Direct Reports Count
0,Bill Lumbergh,NaN,3
1,Bob Slydell,Bill Lumbergh,0
2,Bob Porter,Bill Lumbergh,0
3,Linda M. Grayson,Bill Lumbergh,4
4,Dom Portwood,Linda M. Grayson,3


In [4]:
def get_total_reports(emp):
    if emp not in mgr_to_rep:
        return 0
    total = len(mgr_to_rep[emp])
    for rep in mgr_to_rep[emp]:
        total += get_total_reports(rep)
    return total

df["Total Reports Count"] = df["Employee Name"].apply(get_total_reports)
df.head()

,Employee Name,Manager Name,Direct Reports Count,Total Reports Count
0,Bill Lumbergh,NaN,3,24
1,Bob Slydell,Bill Lumbergh,0,0
2,Bob Porter,Bill Lumbergh,0,0
3,Linda M. Grayson,Bill Lumbergh,4,21
4,Dom Portwood,Linda M. Grayson,3,8


In [10]:
def get_heirarchy(emp):
    hierarchy = []
    while emp is not None:
        hierarchy.append(emp)
        emp = emp_to_mgr[emp]
    return ">".join(hierarchy[::-1])

df["Hierarchy"] = df["Employee Name"].apply(get_heirarchy)
df.head()

,Employee Name,Manager Name,Direct Reports Count,Total Reports Count,Hierarchy
0,Bill Lumbergh,NaN,3,24,Bill Lumbergh
1,Bob Slydell,Bill Lumbergh,0,0,Bill Lumbergh>Bob Slydell
2,Bob Porter,Bill Lumbergh,0,0,Bill Lumbergh>Bob Porter
3,Linda M. Grayson,Bill Lumbergh,4,21,Bill Lumbergh>Linda M. Grayson
4,Dom Portwood,Linda M. Grayson,3,8,Bill Lumbergh>Linda M. Grayson>Dom Portwood


In [11]:
df["Total Reports Count"].sum()

np.int64(70)